In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np

class FloodSegmentationDataset(Dataset):
    def __init__(self, root_dir, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.img_dir = os.path.join(root_dir, "images")
        self.mask_dir = os.path.join(root_dir, "masks")
        self.transform = transform
        self.target_transform = target_transform

        self.images = sorted(os.listdir(self.img_dir))
        self.masks  = sorted(os.listdir(self.mask_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
      img_path  = os.path.join(self.img_dir, self.images[idx])
      mask_path = os.path.join(self.mask_dir, self.masks[idx])

      image = Image.open(img_path).convert("RGB")
      mask  = Image.open(mask_path).convert("L")

      if self.transform:
          image = self.transform(image)

      if self.target_transform:
          mask = self.target_transform(mask)
      else:
          mask = torch.from_numpy(np.array(mask)).long()

      mask = remap_mask(mask)

      return image, mask


In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import os
import numpy as np
from torch.utils.data import DataLoader, Subset

# Define transforms for images and masks
image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Keep segmentation masks intact
    transforms.PILToTensor(),

])



# Root folder that contains: images/ and masks/
root_dir = os.path.join(path, "dataset")

# List image filenames
all_files = sorted(os.listdir(os.path.join(root_dir, "images")))

# Split 80% train, 20% test
train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42, shuffle=True)

full_dataset = FloodSegmentationDataset(root_dir=root_dir,
                                       transform=image_transforms,
                                       target_transform=mask_transforms)

# Convert filenames
file_to_idx = {fname: i for i, fname in enumerate(all_files)}
train_idx = [file_to_idx[f] for f in train_files]
test_idx  = [file_to_idx[f] for f in test_files]

# Create train/test subsets
train_dataset = Subset(full_dataset, train_idx)
test_dataset  = Subset(full_dataset, test_idx)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

print(f"Training Samples: {len(train_dataset)}, Testing Samples: {len(test_dataset)}")


In [ ]:
import matplotlib.pyplot as plt

def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img = img.numpy().transpose(1, 2, 0)
    img = img * std + mean
    img = np.clip(img, 0, 1)
    return img

# Display images + masks
for i in range(3):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(denormalize(img))
    axes[0].set_title("Image")
    axes[0].axis("off")

    axes[1].imshow(mask.squeeze(0).numpy(), cmap="gray")  # <-- FIX
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")

    plt.show()



In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

device = "cpu"  # or "cuda"

model = smp.Unet(
    encoder_name="efficientnet-b1",   # ← changed from b0 → b1
    encoder_weights="imagenet",
    in_channels=3,
    classes=8
).to(device)


In [ ]:
# TO DO
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).to(torch.float)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).to(torch.float)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_pixels = 0

    for images, masks in loader:
        images = images.to(device)
        masks  = masks.to(device).long()

        # if mask is (B,1,H,W) -> (B,H,W)
        if masks.dim() == 4 and masks.size(1) == 1:
            masks = masks.squeeze(1)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        bs = images.size(0)
        total_loss += loss.item() * bs
        total_pixels += bs

    return total_loss / total_pixels


with torch.no_grad():
  def validate(model, loader, criterion, device):
      model.eval()
      total_loss = 0.0
      total_pixels = 0

      for images, masks in loader:
          images = images.to(device)
          masks  = masks.to(device).long()

          if masks.dim() == 4 and masks.size(1) == 1:
              masks = masks.squeeze(1)

          outputs = model(images)
          loss = criterion(outputs, masks)

          bs = images.size(0)
          total_loss += loss.item() * bs
          total_pixels += bs

      return total_loss / total_pixels


num_epochs = 10
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss   = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

# Plot loss curve
plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()


In [ ]:
# TO DO
import random
import matplotlib.pyplot as plt
import numpy as np

def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img = img.numpy().transpose(1, 2, 0)
    img = img * std + mean
    img = np.clip(img, 0, 1)
    return img

model.eval()

test_samples = random.sample(range(len(test_dataset)), 5)

for idx in test_samples:
    img, mask = test_dataset[idx]

    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(device))   # (1, 8, H, W)
        pred_mask = logits.argmax(dim=1).cpu().squeeze(0).numpy()  # (H, W)

    gt_mask = mask
    if isinstance(gt_mask, torch.Tensor):
        if gt_mask.dim() == 3:        # (1,H,W)
            gt_mask = gt_mask.squeeze(0)
        gt_mask = gt_mask.cpu().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(denormalize(img))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(gt_mask, cmap="gray")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    axes[2].imshow(pred_mask, cmap="gray")
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()
